# 관측형 GTFS 경로탐색 — 서울 동대문구 (r5py)

실시간 도착정보 API로 구축한 **관측형 GTFS**를 다중경로 탐색엔진(**r5py / R5**)에 넣어,
동대문구 내 **출발지 → 도착지** 여정을 탐색·지도화한다. 생성된 지도를 캡처해 포스터에 활용.

**실행 환경**
- r5py(JDK21 필요) 설치 환경에서 실행 — 로컬 conda env 예: `r5py`, `JAVA_HOME=.../jdk21`.
- 서버에서 수집한 GTFS를 쓰려면: 서버에서 `gtfs_output/<날짜>/`를 push → 로컬에서 pull.
- 서버의 해당 서비스일 피드는 최신 코드로 `reprocess <날짜>` 하여 R5 호환(빈 stop_id 제거) 상태여야 함.
- 서울 OSM(pbf)은 없으면 셀1이 BBBike에서 자동 다운로드(~52MB).


In [ ]:
import os
# ── Java 환경 (r5py = JDK21). r5py import 전에 설정 ─────────────────────────
os.environ.setdefault("JAVA_HOME", "/Users/chaelyn/miniforge3/envs/jdk21/lib/jvm")  # 환경에 맞게 수정

from pathlib import Path
import urllib.request

ROOT = Path.cwd()
if not (ROOT / "gtfs_output").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
print("ROOT =", ROOT)

# ── 설정 (여기만 수정) ──────────────────────────────────────────────────────
SERVICE_DATE = "20260916"                              # 관측 GTFS 서비스일 (full-day 권장)
ORIGIN       = ("방학사거리",   37.662980, 127.040387)   # 130번만 경유(도봉)
DESTINATION  = ("상암고등학교", 37.581924, 126.884200)   # 271번만 경유(마포 상암)
DEPARTURE    = "10:00:00"                               # 서비스일 기준 출발 시각
PREFER_TRANSFER = True                                  # 환승(2개 이상 노선) 경로 우선 선택
# 팁: 파일럿은 3개 노선(271·272·130)만 있음. 위 O/D는 130→(제기동 환승)→271 이 강제됨.
#     271↔272 환승은 중랑교(공유정류장). 단일노선 경로는 O/D를 같은 노선 위에 두면 됨.
# ───────────────────────────────────────────────────────────────────────────

GTFS_ZIP = ROOT / f"gtfs_output/{SERVICE_DATE}/seoul_dongdaemun_gtfs_{SERVICE_DATE}.zip"
assert GTFS_ZIP.exists(), f"GTFS zip 없음: {GTFS_ZIP}  (해당 서비스일 수집·생성 필요)"

# 서울 OSM (없으면 BBBike에서 자동 다운로드, ~52MB)
OSM_PBF = ROOT / "data/osm/Seoul.osm.pbf"
OSM_PBF.parent.mkdir(parents=True, exist_ok=True)
if (not OSM_PBF.exists()) or OSM_PBF.stat().st_size < 40_000_000:
    print("서울 OSM pbf 다운로드 중… (~52MB, 1회)")
    urllib.request.urlretrieve("https://download.bbbike.org/osm/bbbike/Seoul/Seoul.osm.pbf", OSM_PBF)
print("GTFS =", GTFS_ZIP.name, "| OSM =", OSM_PBF.name)
print(f"{ORIGIN[0]} → {DESTINATION[0]} | 출발 {SERVICE_DATE} {DEPARTURE}")

In [ ]:
%%time
# 관측형 GTFS + 서울 도로망으로 R5 네트워크 빌드 → 상세 경로 탐색
import datetime
import geopandas as gpd
from shapely.geometry import Point
from r5py import TransportNetwork, DetailedItineraries, TransportMode

net = TransportNetwork(str(OSM_PBF), [str(GTFS_ZIP)])
o = gpd.GeoDataFrame({"id": [ORIGIN[0]]},      geometry=[Point(ORIGIN[2], ORIGIN[1])],           crs="EPSG:4326")
d = gpd.GeoDataFrame({"id": [DESTINATION[0]]}, geometry=[Point(DESTINATION[2], DESTINATION[1])], crs="EPSG:4326")
dep = datetime.datetime.strptime(f"{SERVICE_DATE} {DEPARTURE}", "%Y%m%d %H:%M:%S")

di = DetailedItineraries(net, origins=o, destinations=d, departure=dep,
        transport_modes=[TransportMode.TRANSIT, TransportMode.WALK], snap_to_network=True,
        max_time=datetime.timedelta(hours=3)).copy()

# 옵션별 요약: 이동/대기 합, 사용한 버스 노선 수(nroute)
di["tt_min"]   = di["travel_time"].dt.total_seconds() / 60
di["wait_min"] = di["wait_time"].dt.total_seconds() / 60
opt = (di.groupby("option")
         .agg(ride=("tt_min", "sum"), wait=("wait_min", "sum"),
              nroute=("route_id", lambda s: s.dropna().nunique()),
              has_bus=("transport_mode", lambda s: s.astype(str).str.contains("BUS").any()))
         .reset_index())
opt["trip_min"] = opt.ride + opt.wait

# 선택: PREFER_TRANSFER면 2개 이상 노선(환승) 중 최단, 없으면 버스 포함 최단
if PREFER_TRANSFER and (opt.nroute >= 2).any():
    cand = opt[opt.nroute >= 2].sort_values("trip_min")
else:
    cand = opt[opt.has_bus].sort_values("trip_min")
best = int((cand if not cand.empty else opt.sort_values("trip_min")).iloc[0]["option"])
journey = di[di.option == best].sort_values("segment").reset_index(drop=True)
_nr = int(opt.set_index("option").loc[best, "nroute"])
print(f"선택 여정 option={best} | 총 {opt.set_index('option').loc[best,'trip_min']:.0f}분 "
      f"(이동 {journey.tt_min.sum():.0f} + 대기 {journey.wait_min.sum():.0f}) | 버스 노선 {_nr}개 | 구간 {len(journey)}개")

In [ ]:
# 구간표 — 수단·승하차 정류장·시각
import io, zipfile
import pandas as pd

with zipfile.ZipFile(GTFS_ZIP) as z:
    _rt = pd.read_csv(io.BytesIO(z.read("routes.txt")), dtype=str)
    _sp = pd.read_csv(io.BytesIO(z.read("stops.txt")),  dtype=str)
RN = dict(zip(_rt.route_id, _rt.route_short_name))
SN = dict(zip(_sp.stop_id, _sp.stop_name))

rows = []
for _, g in journey.iterrows():
    walk = "WALK" in str(g["transport_mode"])
    dt = g["departure_time"]
    board = "" if pd.isna(dt) else pd.to_datetime(dt).strftime("%H:%M")
    arr   = "" if pd.isna(dt) else (pd.to_datetime(dt) + g["travel_time"]).strftime("%H:%M")
    rows.append({
        "구간": int(g["segment"]),
        "수단": "도보" if walk else f"버스 {RN.get(str(g['route_id']), g['route_id'])}",
        "승차정류장": "" if walk else SN.get(str(g["start_stop_id"]), g["start_stop_id"]),
        "하차정류장": "" if walk else SN.get(str(g["end_stop_id"]),   g["end_stop_id"]),
        "출발": board, "대기(분)": round(g["wait_min"], 1),
        "이동(분)": round(g["tt_min"], 1), "도착": arr, "거리(m)": int(g["distance"]),
    })
pd.DataFrame(rows)

In [ ]:
# folium 지도 (참고 make_leg_map 스타일) — 노선 배지·검은 승하차점·역명 라벨·여정 범례
import folium

STOP_LABEL_MIN_ZOOM = 12
PALETTE = ['#d62728','#1f77b4','#2ca02c','#9467bd','#ff7f0e','#17becf','#8c564b','#e377c2','#bcbd22','#7f7f7f']
def stop_nm(s): return SN.get(str(s), str(s))

tr_idx = [i for i, l in journey.iterrows() if "WALK" not in str(l["transport_mode"])]
leg_color = {i: PALETTE[k % len(PALETTE)] for k, i in enumerate(tr_idx)}
J = journey[journey.geometry.notna()]
_c = J.geometry.union_all().centroid
m = folium.Map(location=[_c.y, _c.x], zoom_start=13, tiles="OpenStreetMap")

def _station_label(lat, lon, text, above):
    shift = 'translate(-50%,-165%)' if above else 'translate(-50%,60%)'
    folium.Marker([lat, lon], icon=folium.DivIcon(icon_size=(0,0), icon_anchor=(0,0), html=(
        f'<div class="stop-label" style="background:#fff;border:2px solid #000;color:#000;'
        f'padding:4px 18px;border-radius:5px;font-size:14px;font-weight:bold;white-space:nowrap;'
        f'display:none;text-align:center;transform:{shift};box-shadow:0 1px 3px rgba(0,0,0,.3);">{text}</div>'))).add_to(m)
def _black_dot(lat, lon, tip):
    folium.CircleMarker([lat, lon], radius=8, color='#000', weight=3, fill=True,
                        fill_color='#000', fill_opacity=1, tooltip=tip).add_to(m)

for i, leg in journey.iterrows():
    if leg.geometry is None:
        continue
    walk = i not in leg_color
    no = RN.get(str(leg.get("route_id", "")), str(leg.get("route_id", "")))
    color = '#666666' if walk else leg_color[i]
    dt = leg["departure_time"]; bt = None if pd.isna(dt) else pd.to_datetime(dt)
    at = None if bt is None else bt + leg["travel_time"]
    if walk:
        tip = f"[{int(leg['segment'])}] 도보 {leg['tt_min']:.0f}분 ({int(leg['distance'])}m)"
    else:
        tip = (f"[{int(leg['segment'])}] {no} · {bt:%H:%M} 승차(대기 {leg['wait_min']:.0f}분) → "
               f"{at:%H:%M} 하차 | {stop_nm(leg['start_stop_id'])} → {stop_nm(leg['end_stop_id'])}")
    folium.GeoJson(leg.geometry.__geo_interface__,
        style_function=lambda f, _c=color, _w=walk: {
            'color': _c, 'weight': 3 if _w else 6, 'opacity': 0.9, 'dashArray': '6' if _w else None},
        tooltip=tip).add_to(m)
    if not walk:
        mid = leg.geometry.interpolate(0.5, normalized=True)
        folium.Marker([mid.y, mid.x], icon=folium.DivIcon(icon_size=(0,0), icon_anchor=(0,0), html=(
            f'<div style="background:{color};color:#fff;padding:5px 14px;border-radius:14px;font-size:18px;'
            f'font-weight:bold;box-shadow:0 1px 5px rgba(0,0,0,.45);white-space:nowrap;display:inline-block;'
            f'text-align:center;transform:translate(-50%,-50%);">{no}</div>'))).add_to(m)
        bx, by = leg.geometry.coords[0]
        _black_dot(by, bx, f"{no} 승차 {stop_nm(leg['start_stop_id'])} ({bt:%H:%M}·대기 {leg['wait_min']:.0f}분)")
        _station_label(by, bx, f"승차 {stop_nm(leg['start_stop_id'])}", False)
        ax, ay = leg.geometry.coords[-1]
        _black_dot(ay, ax, f"{no} 하차 {stop_nm(leg['end_stop_id'])} ({at:%H:%M})")
        _station_label(ay, ax, f"하차 {stop_nm(leg['end_stop_id'])}", True)

_timed = journey[journey.departure_time.notna()]
s_t = pd.to_datetime(_timed.iloc[0]["departure_time"])
e_t = pd.to_datetime(_timed.iloc[-1]["departure_time"]) + _timed.iloc[-1]["travel_time"]
total_min = (e_t - s_t).total_seconds() / 60
_first, _last = J.iloc[0].geometry, J.iloc[-1].geometry
folium.Marker([_first.coords[0][1], _first.coords[0][0]], tooltip=f"출발: {ORIGIN[0]} ({s_t:%H:%M})",
              icon=folium.Icon(color='black', icon='home')).add_to(m)
folium.Marker([_last.coords[-1][1], _last.coords[-1][0]], tooltip=f"도착: {DESTINATION[0]} ({e_t:%H:%M})",
              icon=folium.Icon(color='red', icon='flag')).add_to(m)

rows = []
for i, leg in journey.iterrows():
    dt = leg["departure_time"]; bt = None if pd.isna(dt) else pd.to_datetime(dt)
    at = None if bt is None else bt + leg["travel_time"]
    if i in leg_color:
        no = RN.get(str(leg['route_id']), str(leg['route_id']))
        chip = (f'<span style="display:inline-block;width:11px;height:11px;background:{leg_color[i]};'
                f'border-radius:2px;margin-right:5px;"></span>')
        rows.append(f'<tr><td style="white-space:nowrap;">{chip}<b>{no}</b></td>'
                    f'<td>{bt:%H:%M} {stop_nm(leg["start_stop_id"])} 승차<br>'
                    f'대기 {leg["wait_min"]:.0f}분 · 이동 {leg["tt_min"]:.0f}분<br>'
                    f'{at:%H:%M} {stop_nm(leg["end_stop_id"])} 하차</td></tr>')
    elif leg["tt_min"] >= 0.5:
        rows.append(f'<tr><td style="color:#666;">도보</td>'
                    f'<td style="color:#666;">{leg["tt_min"]:.0f}분 ({int(leg["distance"])}m)</td></tr>')
legend = f'''<div style="position:fixed;top:12px;right:12px;z-index:9999;background:rgba(255,255,255,.96);
  border:2px solid #333;border-radius:8px;padding:10px 12px;font-size:12px;max-height:85vh;overflow-y:auto;
  max-width:290px;box-shadow:0 2px 10px rgba(0,0,0,.35);line-height:1.45;font-family:sans-serif;">
  <div style="font-size:14px;font-weight:bold;">{ORIGIN[0]} → {DESTINATION[0]}</div>
  <div style="margin:3px 0 6px;"><b>출발 {s_t:%H:%M}</b> → <b>도착 {e_t:%H:%M}</b> (총 {total_min:.0f}분)</div>
  <div style="color:#888;margin-bottom:4px;">관측형 GTFS {SERVICE_DATE} · r5py 경로탐색</div>
  <table style="border-collapse:collapse;"><style>td{{padding:3px 4px;border-top:1px solid #ddd;vertical-align:top;}}</style>
  {"".join(rows)}</table><div style="margin-top:6px;color:#444;">● 검은 점 = 승·하차 정류장</div></div>'''
m.get_root().html.add_child(folium.Element(legend))
_mn = m.get_name()
m.get_root().script.add_child(folium.Element(
  f'document.addEventListener("DOMContentLoaded",function(){{function sl(){{var show={_mn}.getZoom()>={STOP_LABEL_MIN_ZOOM};'
  f'document.querySelectorAll(".stop-label").forEach(function(el){{el.style.display=show?"inline-block":"none";}});}}'
  f'{_mn}.on("zoomend",sl);sl();}});'))
out = ROOT / f"gtfs_output/{SERVICE_DATE}/journey_{ORIGIN[0]}_{DESTINATION[0]}.html"
m.save(str(out)); print("저장:", out, "  ← 브라우저로 열어 캡처(포스터용)")
m